# O-information Estimator Comparison

This notebook compares three O-information estimators across three synthetic systems:

- **KDE** (Kernel Density Estimation): a precise Shannon entropy estimator built with `scipy.stats.gaussian_kde` (Scott's bandwidth rule). Returns values in **nats**.
- **HOI-GC** (Gaussian Copula): from the `hoi` library, fast parametric estimator that copula-normalises the data.
- **HOI-KSG** (k-nearest-neighbour / Kozachenko-Leonenko): from the `hoi` library, non-parametric estimator.

The three systems being analysed are:
1. **ReLU system** (`generate_relu_sistem`) – a 4-variable system with explicit synergistic and redundant sources.
2. **Flat system** (`generate_flat_system`) – a 6-variable system where all variables share two latent common causes.
3. **Continuous XOR system** (`generate_continuos_xor`) – a 3-variable system with a synergistic target.

All HOI results (bits) are converted to **nats** by multiplying by $\ln 2$ for a fair comparison.

## 1. Imports and configuration

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy.stats import gaussian_kde
from tqdm.notebook import tqdm

sys.path.insert(0, os.path.join('..', 'benchmarking'))
from systems import generate_flat_system, generate_relu_sistem, generate_continuos_xor

from hoi.metrics import Oinfo

sns.set_context('paper', font_scale=1.3)
plt.rcParams.update({'figure.dpi': 100})

In [ ]:
# ── Experiment parameters (reduce n_repeat / T for a quick run) ───────────
T         = 1000   # samples per dataset
n_repeat  = 5      # repetitions for estimating mean ± std
pow_factor = 0.5   # ReLU system: power applied to smoothSoftPlus
SEED      = 42

alpha_range = np.round(np.linspace(0.0, 1.0, 11), 2)  # 0.0 … 1.0
LOG2 = np.log(2)   # bits → nats conversion factor

ESTIMATOR_COLORS = {'KDE': '#2196F3', 'HOI-GC': '#FF9800', 'HOI-KSG': '#4CAF50'}
ESTIMATOR_MARKERS = {'KDE': 'o', 'HOI-GC': 's', 'HOI-KSG': '^'}

## 2. Entropy and O-information estimators

In [ ]:
# ── KDE Shannon entropy ────────────────────────────────────────────────────

def kde_entropy(X: np.ndarray) -> float:
    """
    Estimate Shannon differential entropy with Gaussian KDE (Scott's rule).

    Uses the plug-in estimator:  H(X) ≈ −1/N Σᵢ log p̂(xᵢ)
    where p̂ is the Gaussian KDE fitted to the data.

    Parameters
    ----------
    X : ndarray of shape (n_samples, n_features)

    Returns
    -------
    float : estimated entropy in nats
    """
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    kde = gaussian_kde(X.T, bw_method='scott')
    return -float(np.mean(kde.logpdf(X.T)))


def o_information_kde(X: np.ndarray) -> float:
    """
    O-information via KDE entropy.

    Ω(X) = (N−2)·H(X) + Σⱼ [H(Xⱼ) − H(X₋ⱼ)]

    Parameters
    ----------
    X : ndarray of shape (n_samples, n_variables)

    Returns
    -------
    float : O-information in nats
    """
    N = X.shape[1]
    mask = (np.ones((N, N)) - np.eye(N)).astype(bool)  # True everywhere except diagonal
    h_joint    = kde_entropy(X)
    h_marginal = np.array([kde_entropy(X[:, [i]]) for i in range(N)])
    h_excl     = np.array([kde_entropy(X[:, idxs]) for idxs in mask])
    return (N - 2) * h_joint + (h_marginal - h_excl).sum()

In [ ]:
# ── HOI batch O-information ────────────────────────────────────────────────

def compute_oinfo_all_estimators(
    X: np.ndarray,
    multiplets: list,
) -> pd.DataFrame:
    """
    Compute O-information for a list of multiplets using all three estimators.

    HOI processes all multiplets in a single JAX-accelerated scan, so GC and
    KSG are computed batch-wise for efficiency. Results are converted to nats.

    Parameters
    ----------
    X          : (n_samples, n_features) array with ALL variables
    multiplets : list of tuples with column indices into X

    Returns
    -------
    DataFrame with columns: nplet, KDE, HOI-GC, HOI-KSG
    """
    min_sz = min(len(m) for m in multiplets)
    max_sz = max(len(m) for m in multiplets)

    # KDE  (computed per-multiplet on the relevant column subset)
    kde_values = [
        o_information_kde(X[:, list(mp)]) for mp in multiplets
    ]

    # HOI batch — one model object, two method calls
    model = Oinfo(X, multiplets=list(multiplets))
    gc_raw  = model.fit(minsize=min_sz, maxsize=max_sz, method='gc').flatten()
    knn_raw = model.fit(minsize=min_sz, maxsize=max_sz, method='knn').flatten()

    return pd.DataFrame({
        'nplet'  : [str(m) for m in multiplets],
        'KDE'    : kde_values,
        'HOI-GC' : [float(v) * LOG2 for v in gc_raw],
        'HOI-KSG': [float(v) * LOG2 for v in knn_raw],
    })

### Quick sanity check on known Gaussians

For $X \sim \mathcal{N}(0, I_N)$ the O-information is exactly 0 (all variables independent). We verify that all estimators recover this.

In [ ]:
np.random.seed(SEED)
X_check = np.random.normal(0, 1, (2000, 4))
mp_check = [(0, 1, 2, 3)]
sanity = compute_oinfo_all_estimators(X_check, mp_check)
print('O-info for independent N(0,I₄)  (expected ≈ 0 nats):')
print(sanity[['KDE','HOI-GC','HOI-KSG']].to_string(index=False))

## 3. Utility: run experiment and collect results

In [ ]:
def run_experiment(generate_fn, multiplet_spec, alpha_range, n_repeat, T, **gen_kwargs):
    """
    Run O-information estimation for a range of alpha values.

    Parameters
    ----------
    generate_fn   : callable(alpha, T, **gen_kwargs) -> DataFrame
    multiplet_spec: dict  {nplet_name: (col_indices_tuple, col_names_list)}
    alpha_range   : 1-D array of alpha values
    n_repeat      : int, number of realisations per alpha
    T             : int, samples per realisation
    gen_kwargs    : extra keyword arguments forwarded to generate_fn

    Returns
    -------
    DataFrame with columns: alpha, nplet, KDE, HOI-GC, HOI-KSG, repeat
    """
    rows = []
    multiplets = [spec[0] for spec in multiplet_spec.values()]
    nplet_names = list(multiplet_spec.keys())

    for alpha in tqdm(alpha_range, desc='alpha', leave=True):
        for rep in range(n_repeat):
            data = generate_fn(alpha=alpha, T=T, **gen_kwargs)
            X_all = data.values.astype(float)
            df_rep = compute_oinfo_all_estimators(X_all, multiplets)
            df_rep['nplet'] = nplet_names  # use human-readable names
            df_rep['alpha'] = alpha
            df_rep['repeat'] = rep
            rows.append(df_rep)

    return pd.concat(rows, ignore_index=True)

In [ ]:
def plot_system_results(results: pd.DataFrame, title: str, fig_path: str = None):
    """
    Plot O-information vs alpha for each nplet, comparing all three estimators.

    One subplot per unique nplet; lines are estimator means ± 1 std across repeats.
    """
    nplets = results['nplet'].unique()
    n_cols = min(len(nplets), 3)
    n_rows = int(np.ceil(len(nplets) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(5 * n_cols, 4 * n_rows),
                             sharex=True, sharey=False)
    axes = np.array(axes).flatten()

    for ax, nplet in zip(axes, nplets):
        sub = results[results['nplet'] == nplet]
        for estimator, color in ESTIMATOR_COLORS.items():
            grp = sub.groupby('alpha')[estimator]
            mu  = grp.mean()
            std = grp.std()
            ax.plot(mu.index, mu.values,
                    color=color, marker=ESTIMATOR_MARKERS[estimator],
                    markersize=4, label=estimator)
            ax.fill_between(mu.index, mu - std, mu + std,
                            color=color, alpha=0.15)
        ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
        ax.set_title(nplet, fontsize=10)
        ax.set_xlabel(r'$\alpha$')
        ax.set_ylabel(r'$\Omega$ (nats)')
        ax.grid(True, linewidth=0.4)

    # Hide unused axes
    for ax in axes[len(nplets):]:
        ax.set_visible(False)

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=3,
               bbox_to_anchor=(0.5, 1.02), frameon=True)
    fig.suptitle(title, fontsize=14, y=1.05)
    plt.tight_layout()

    if fig_path:
        os.makedirs(os.path.dirname(fig_path), exist_ok=True)
        plt.savefig(fig_path, bbox_inches='tight')
    plt.show()

## 4. ReLU system

The **ReLU system** (`generate_relu_sistem`) has four variables:

$$
X_1 = \alpha \cdot [\text{softplus}(Z_{\text{syn}})]^p + \beta Z_{\text{red}}, \qquad
X_2 = -\alpha \cdot [\text{softplus}(-Z_{\text{syn}})]^p + \beta Z_{\text{red}}
$$

with $\beta = 1 - \alpha$. $Z_{\text{syn}}$ is a synergistic source shared by $X_1$ and $X_2$ (appearing with opposite signs), while $Z_{\text{red}}$ is a redundant source (appearing with the same sign). The parameter $\alpha$ controls the balance: at $\alpha=0$ the system is purely redundant, at $\alpha=1$ it is purely synergistic.

We evaluate five n-plets of interest.

In [ ]:
# Column layout of generate_relu_sistem output: X1=0, X2=1, Z_syn=2, Z_red=3
RELU_MULTIPLETS = {
    'X1-X2-Zsyn-Zred': ((0, 1, 2, 3), ['X1','X2','Z_syn','Z_red']),
    'X1-X2-Zsyn'     : ((0, 1, 2),    ['X1','X2','Z_syn']),
    'X1-X2-Zred'     : ((0, 1, 3),    ['X1','X2','Z_red']),
    'X1-Zred-Zsyn'   : ((0, 2, 3),    ['X1','Z_syn','Z_red']),
    'X2-Zred-Zsyn'   : ((1, 2, 3),    ['X2','Z_syn','Z_red']),
}

In [ ]:
def generate_relu_for_experiment(alpha, T, pow_factor=0.5):
    """Wrapper so beta = 1 - alpha is enforced automatically."""
    return generate_relu_sistem(alpha=alpha, beta=1-alpha, pow_factor=pow_factor, T=T)

np.random.seed(SEED)
print(f'Running ReLU system: T={T}, n_repeat={n_repeat}, pow_factor={pow_factor}')
relu_results = run_experiment(
    generate_fn    = generate_relu_for_experiment,
    multiplet_spec = RELU_MULTIPLETS,
    alpha_range    = alpha_range,
    n_repeat       = n_repeat,
    T              = T,
    pow_factor     = pow_factor,
)
relu_results.head(10)

In [ ]:
plot_system_results(
    relu_results,
    title=f'ReLU system  (T={T}, repeats={n_repeat}, pow={pow_factor})',
    fig_path='./figures/estimators/relu_estimator_comparison.pdf',
)

## 5. Flat system

The **flat system** (`generate_flat_system`) has six observed variables $X_1 \ldots X_6$ each built from two shared latent sources:

$$
X_k = \alpha \cdot f_k(Z_{00}) + \beta \cdot Z_{01} + \gamma Z_k
$$

where $f_k$ are distinct nonlinear transforms of the same source $Z_{00}$, and $Z_{01}$ is another shared term. $\gamma = 0.1$ is fixed; $\beta = 1 - \alpha$. Because all $X_k$ share **both** $Z_{00}$ and $Z_{01}$, we expect positive (redundant) O-information that grows with $\alpha$ and $\beta$.

We select a few representative n-plets.

In [ ]:
# Column layout from generate_flat_system:
# X1=0, X2=1, X3=2, X4=3, X5=4, X6=5, Z1=6..Z6=11, Z00=12, Z01=13

FLAT_MULTIPLETS = {
    'X1-X2-X3'            : ((0, 1, 2),           None),
    'X1-X2-X3-X4'         : ((0, 1, 2, 3),         None),
    'X1-X2-X3-X4-X5'      : ((0, 1, 2, 3, 4),      None),
    'X1-X2-X3-X4-X5-X6'   : ((0, 1, 2, 3, 4, 5),   None),
    'Z00-X1-X2-X3'         : ((12, 0, 1, 2),        None),
    'Z01-X1-X2-X3'         : ((13, 0, 1, 2),        None),
}

In [ ]:
def generate_flat_for_experiment(alpha, T, gamma=0.1):
    """Wrapper so beta = 1 - alpha is enforced and gamma is fixed."""
    return generate_flat_system(alpha=alpha, beta=1-alpha, gamma=gamma, T=T)

np.random.seed(SEED)
print(f'Running Flat system: T={T}, n_repeat={n_repeat}, gamma=0.1')
flat_results = run_experiment(
    generate_fn    = generate_flat_for_experiment,
    multiplet_spec = FLAT_MULTIPLETS,
    alpha_range    = alpha_range,
    n_repeat       = n_repeat,
    T              = T,
    gamma          = 0.1,
)
flat_results.head()

In [ ]:
plot_system_results(
    flat_results,
    title=f'Flat system  (T={T}, repeats={n_repeat}, γ=0.1, β=1−α)',
    fig_path='./figures/estimators/flat_estimator_comparison.pdf',
)

## 6. Continuous XOR system

The **continuous XOR system** (`generate_continuos_xor`) has three variables:

$$
Z_{\text{xor}} = \alpha(Z + 4 \cdot \mathbb{1}[X_1 > 0] \oplus \mathbb{1}[X_2 > 0]) + (1-\alpha) Z
$$

$Z_{\text{xor}}$ encodes the XOR of the signs of $X_1$ and $X_2$ scaled by $\alpha$. At $\alpha = 0$ all three variables are independent (O-information ≈ 0). As $\alpha \to 1$, the system becomes more synergistic (O-information should be increasingly negative).

In [ ]:
# Column layout: X1=0, X2=1, Zxor=2
XOR_MULTIPLETS = {
    'X1-X2-Zxor': ((0, 1, 2), None),
}

In [ ]:
def generate_xor_for_experiment(alpha, T):
    return generate_continuos_xor(alpha=alpha, T=T)

np.random.seed(SEED)
print(f'Running XOR system: T={T}, n_repeat={n_repeat}')
xor_results = run_experiment(
    generate_fn    = generate_xor_for_experiment,
    multiplet_spec = XOR_MULTIPLETS,
    alpha_range    = alpha_range,
    n_repeat       = n_repeat,
    T              = T,
)
xor_results.head()

In [ ]:
plot_system_results(
    xor_results,
    title=f'Continuous XOR system  (T={T}, repeats={n_repeat})',
    fig_path='./figures/estimators/xor_estimator_comparison.pdf',
)

## 7. Combined summary plot

One row per system; each column is a representative n-plet that captures the key feature of that system.

In [ ]:
SUMMARY_SPEC = [
    ('ReLU – full system',  relu_results, 'X1-X2-Zsyn-Zred'),
    ('ReLU – synergistic',  relu_results, 'X1-X2-Zsyn'),
    ('ReLU – redundant',    relu_results, 'X1-X2-Zred'),
    ('Flat – 3 vars',       flat_results, 'X1-X2-X3'),
    ('Flat – 6 vars',       flat_results, 'X1-X2-X3-X4-X5-X6'),
    ('Flat – Z00 added',    flat_results, 'Z00-X1-X2-X3'),
    ('XOR',                 xor_results,  'X1-X2-Zxor'),
]

n_panels = len(SUMMARY_SPEC)
n_cols = 4
n_rows = int(np.ceil(n_panels / n_cols))

fig, axes = plt.subplots(n_rows, n_cols,
                          figsize=(5 * n_cols, 4 * n_rows),
                          sharex=True)
axes = np.array(axes).flatten()

for ax, (subtitle, df, nplet_name) in zip(axes, SUMMARY_SPEC):
    sub = df[df['nplet'] == nplet_name]
    for estimator, color in ESTIMATOR_COLORS.items():
        grp = sub.groupby('alpha')[estimator]
        mu  = grp.mean()
        std = grp.std()
        ax.plot(mu.index, mu.values,
                color=color, marker=ESTIMATOR_MARKERS[estimator],
                markersize=4, label=estimator)
        ax.fill_between(mu.index, mu - std, mu + std, color=color, alpha=0.15)
    ax.axhline(0, color='k', linewidth=0.7, linestyle='--')
    ax.set_title(subtitle, fontsize=9, pad=4)
    ax.set_xlabel(r'$\alpha$', fontsize=9)
    ax.set_ylabel(r'$\Omega$ (nats)', fontsize=9)
    ax.grid(True, linewidth=0.4)

for ax in axes[n_panels:]:
    ax.set_visible(False)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', ncol=3,
           bbox_to_anchor=(0.5, 1.01), fontsize=11, frameon=True)
fig.suptitle('O-information estimator comparison across systems', fontsize=14, y=1.04)
plt.tight_layout()

os.makedirs('./figures/estimators', exist_ok=True)
plt.savefig('./figures/estimators/summary_estimator_comparison.pdf', bbox_inches='tight')
plt.show()

## 8. Save results

In [ ]:
os.makedirs('../benchmarking/results/estimators', exist_ok=True)

relu_results.to_csv('../benchmarking/results/estimators/relu_estimator_comparison.tsv', sep='\t', index=False)
flat_results.to_csv('../benchmarking/results/estimators/flat_estimator_comparison.tsv', sep='\t', index=False)
xor_results.to_csv('../benchmarking/results/estimators/xor_estimator_comparison.tsv',  sep='\t', index=False)

print('Results saved.')

## Notes on estimator properties

| Estimator | Type | Units returned | Bias | Variance | Speed |
|-----------|------|---------------|------|----------|-------|
| **KDE** | Non-parametric plug-in | nats | Moderate (bandwidth bias) | Low-moderate | Medium |
| **HOI-GC** | Parametric (Gaussian copula) | bits → nats | Low for Gaussian data | Very low | Fast |
| **HOI-KSG** | Non-parametric (k-NN) | bits → nats | Low | Moderate | Slow |

**KDE precision notes**: The Gaussian KDE with Scott's bandwidth minimises MISE for Gaussian data, but can over-smooth multimodal distributions. For the systems studied here (with non-Gaussian marginals due to the ReLU / XOR transformations), the KDE estimator may underestimate entropy, which translates directly into bias in O-information. Increasing `T` reduces this bias.

**HOI-GC**: The Gaussian Copula estimator fits a Gaussian distribution to the copula-normalised data. It is very fast and has low variance, but it can be biased for distributions with heavy non-Gaussian dependence structures (e.g., the XOR system).

**HOI-KSG**: The KSG estimator is consistent and makes minimal distributional assumptions, but has higher variance and is computationally expensive ($O(N^2)$ in the naive implementation).